In [ ]:
%%capture
%pip install gradio==4.44.0
%pip install --upgrade gradio
%pip install ibm-watsonx-ai==1.1.2
%pip install langchain==0.2.11
%pip install langchain-community==0.2.10
%pip install langchain-ibm==0.1.11
%pip install chromadb==0.4.24
%pip install pypdf==4.3.1
%pip install pydantic==2.9.1
%pip install huggingface_hub==0.23.0
%pip install gradio
%pip install huggingface_hub
%pip install --upgrade chromadb
#%pip install -U ipywidgets
%pip install openai
%pip install sentence-transformers
%pip install jq
%pip install "transformers<4.49.0"
%pip install datasets
%pip install "ragas==0.2.2"
%pip install nltk
%pip install deepeval litellm Ollama
%pip install --force-reinstall numpy==1.26.4

In [ ]:
!sudo apt-get update && sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess
import time

# Start the server in the background
process = subprocess.Popen(['ollama', 'serve'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

# Give the server a few seconds to initialize
time.sleep(5)
print("Ollama server is running in the background!")
!ollama pull llama3

In [ ]:


from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain.chains import RetrievalQA
from huggingface_hub import get_token
import logging
import sys
import warnings
from langchain_community.document_loaders import JSONLoader
import json
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import Ollama
from langchain_core.prompts import PromptTemplate
import gc
import os
import numpy as np
# Dynamically restore the expected alias
if not hasattr(np, 'long'):
    np.long = np.int64
if not hasattr(np, 'ulong'):
    np.ulong = np.uint64
from sentence_transformers import CrossEncoder
from google.colab import drive
drive.mount('/content/drive')

# Configure the root logger
logger = logging.getLogger()
logger.setLevel(logging.INFO)

# Avoid adding duplicate handlers if the script reloads
if not logger.handlers:
    # Create console handler and set level to info
    ch = logging.StreamHandler(sys.stdout)
    ch.setLevel(logging.INFO)

    # Create formatter
    formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
    ch.setFormatter(formatter)

    # Add ch to logger
    logger.addHandler(ch)# Configure the root logger
logger = logging.getLogger()
logger.setLevel(logging.INFO)

# Avoid adding duplicate handlers if the script reloads
if not logger.handlers:
    # Create console handler and set level to info
    ch = logging.StreamHandler(sys.stdout)
    ch.setLevel(logging.INFO)

    # Create formatter
    formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
    ch.setFormatter(formatter)

    # Add ch to logger
    logger.addHandler(ch)

# You can use this section to suppress warnings generated by your code:
def warn(*args, **kwargs):
    pass
warnings.warn = warn
warnings.filterwarnings('ignore')

def append_to_json(file_path, new_data):
    # 1. Check if file exists and is not empty
    if os.path.exists(file_path) and os.path.getsize(file_path) > 0:
        with open(file_path, 'r', encoding='utf-8') as file:
            # Load existing data into a list
            data_list = json.load(file)
    else:
        # If file doesn't exist, start with an empty list
        data_list = []

    # 2. Append your new labeled data to the list
    data_list.append(new_data)

    # 3. Write everything back to the file with formatting
    with open(file_path, 'w', encoding='utf-8') as file:
        json.dump(data_list, file, indent=4)

## Document loader
def document_loader(file):
    if file is None:
        logging.info( "Please upload a valid JSON file.")
    loader = JSONLoader(
        file_path=file,
        jq_schema='.[]', # Adjust the jq_schema based on your JSON structure
        text_content=False
    )
    loaded_document = loader.load()
    return loaded_document

## Text splitter
def text_splitter(data):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=200,
        chunk_overlap=20,
        length_function=len,
    )
    chunks = text_splitter.split_documents(data)
    return chunks

def hf_embedding():
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    model_kwargs = {'device': 'cpu'}
    encode_kwargs = {'normalize_embeddings': False}

    embedding_model = HuggingFaceEmbeddings(
        model_name=model_name
    )
    return embedding_model


## Vector db
def vector_database(chunks):
    embedding_model = hf_embedding()
    vectordb = Chroma.from_documents(chunks, embedding_model)
    return vectordb

def create_vector_db(file):
    if file is None:
        logging.info("Please upload a valid JSON file.")
    splits = document_loader(file)
    chunks = text_splitter(splits)
    vectordb = vector_database(chunks)
    return vectordb.as_retriever()

# 1. Load your input JSON file
# Assumes the JSON file is a list of objects or a dict containing questions
input_file_path = "/content/drive/MyDrive/qa_data_test_Final_50.json"
with open(input_file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# Extract the list of questions (adjust the key based on your JSON structure)
questions = [item["question"] for item in data]
contexts = [item["knowledge"] for item in data]
ground_truths = [item["right_answer"] for item in data]
template = """You are an expert fact-extraction system. You must output EXACTLY the correct answer from the provided context. Do NOT use conversational filler, do NOT add introductory text like 'According to the context', and do NOT output full sentences. Only output the exact answer.
Context: {context}
Question: {question}
Answer:"""

QA_CHAIN_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template=template,
)

llm = Ollama(model="llama3",
    temperature=1.0,
    num_predict=200, # Max output tokens
    num_ctx=4096)
   # keep_alive=0)     # Max total context tokens)


splits = document_loader(input_file_path)
chunks = text_splitter(splits)
vectordb = vector_database(chunks)
label_mapping = ['contradiction', 'entailment', 'neutral']
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectordb.as_retriever(),
    return_source_documents=True
   # chain_type_kwargs={"prompt": QA_CHAIN_PROMPT}
)
nli_model = CrossEncoder('cross-encoder/nli-deberta-v3-base')
for i, (question, context, ground_truth) in enumerate(zip(questions, contexts, ground_truths)):
    try:
        print("I_1:",i)
        response = qa.invoke(question)
        print("After llm call:")
        if isinstance(response, dict):
            answer = response.get("result", "").strip()
        else:
            answer = str(response).strip()
        print(question,";",context,";",ground_truth,";",answer,"\n")

        gt_scores = nli_model.predict([(ground_truth, answer)])
        predicted_label = label_mapping[np.argmax(gt_scores[0])]
       # print(f"Entailment score (Context vs Answer): {entailment_prob:.4f}")
        print(f"Accuracy vs Ground Truth: {predicted_label}")
        output_data = {
            "Temperature": 1.0,
            "Context_length": 4096,
            "Context": context,
            "Question":question,
            "Ground_truth": ground_truth,
            "Answer_by_llm": answer,
            "Predicted_label": predicted_label,
            "Chunk_Size": 200,
            "Prompt": "OFF"
        }
        # Clear memory every 10-20 iterations
        append_to_json('/content/drive/MyDrive/output_202.json', output_data)
        if i%10==0 and i!=0:
          process.terminate()  # Sends SIGTERM
          try:
              process.wait(timeout=5)  # Wait up to 5 seconds for it to close
              print("Ollama server stopped successfully.")
          except subprocess.TimeoutExpired:
              process.kill()  # Force kill if it hangs
              print("Ollama server force killed.")

          # 4. Restart the server
          process = subprocess.Popen(['ollama', 'serve'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
          print("Ollama server restarted.")
          !ollama pull llama3
          print("Ollama pull done.")



    except Exception as e:
        logging.error(f"Error during chain invocation: {e}")

from google.colab import drive
drive.mount('/content/drive')



/tmp/ipykernel_32682/2165430741.py:20: FutureWarning: In the future `np.long` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, 'long'):
/tmp/ipykernel_32682/2165430741.py:22: FutureWarning: In the future `np.ulong` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, 'ulong'):
INFO:sentence_transformers.base.model:No device provided, using cpu


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


INFO:sentence_transformers.base.model:Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
INFO:sentence_transformers.base.model:No device provided, using cpu
INFO:sentence_transformers.base.model:No modules.json found for cross-encoder/nli-deberta-v3-base, initializing a new CrossEncoder model.


I_1: 0
After llm call:
Which magazine was started first Arthur's Magazine or First for Women? ; Arthur's Magazine (1844–1846) was an American literary periodical published in Philadelphia in the 19th century.First for Women is a woman's magazine published by Bauer Media Group in the USA. ; Arthur's Magazine ; Arthur's Magazine 



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Accuracy vs Ground Truth: contradiction
I_1: 48
After llm call:
What actor has appeared in the films "Hamlet", "The Woodlanders" and "The Holiday"? ;  Jude Law and Jack Black were cast as the film's leading men Graham and Miles, with Eli Wallach, Shannyn Sossamon, Edward Burns and Rufus Sewell playing key supporting roles.Rufus Frederik Sewell ( ; born 29 October 1967) is an English actor. In film, he has appeared in Kenneth Branagh's rendition of "Hamlet" (1996) playing Fortinbras, "The Woodlanders", "Dangerous Beauty", "Dark City", "A Knight's Tale", "The Illusionist", "Tristan and Isolde", and "Martha, Meet Frank, Daniel and Laurence". ; Rufus Frederik Sewell ; According to the provided text, Rufus Sewell has appeared in the films "Hamlet", "The Woodlanders", and "The Holiday". 



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Accuracy vs Ground Truth: neutral
I_1: 49
After llm call:
John Stanley Carroll attended which of the ten branches of the University of Hawai'? ;  He is a lawyer, having been educated at Saint Mary's University, the University of Hawaii at Hilo and the University of Hawaii at Manoa. It is one of ten branches of the University of Hawaiʻ i system. ; been educated at Saint Mary's University, the University of Hawaii at Hilo ; University of Hawaii at Manoa 



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Accuracy vs Ground Truth: contradiction
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
